# Data Cleaning 

## Imports

In [176]:
import pandas as pd

## Load CIA data

In [177]:
# Load the original CSV file
df = pd.read_csv("raw/countries.csv")

# List the columns you want to keep
columns_to_keep = [
    "Country",
    "url",
    "Introduction: Background",
    "Geography: Location",
    "Geography: Map references",
    "Geography: Area - comparative"
]

# Filter only the relevant columns
df_clean = df[columns_to_keep]

# (Optional) Rename columns for convenience
df_clean.columns = [
    "country", 
    "url", 
    "background", 
    "location", 
    "map_reference", 
    "area_comparative"
]

# Preview the cleaned DataFrame
df_clean.head()

# Save the cleaned DataFrame to a new CSV file
df_clean.to_csv("raw/countries_clean.csv", index=False)


## Load additional data files

In [178]:
# List all your source files
files = {
    "capitals": "raw/countries_capitals.csv",
    "languages": "raw/country_languages.csv",
    "food": "raw/country_food.csv",
    "flags": "raw/country_flag.csv"
}

# Load and inspect all files
dfs = {}
for key, path in files.items():
    try:
        df = pd.read_csv(path, encoding='utf-8', engine='python', dtype=str)
        print(f"\n🟡 {key.upper()} — HEAD:\n", df.head())
        print(f"Columns: {df.columns.tolist()}, Rows: {len(df)}")
        dfs[key] = df
    except Exception as e:
        print(f"\n🔴 Fehler beim Laden von {path}: {e}")



🟡 CAPITALS — HEAD:
                 country           capital Population Area(Sq.Km)
0               Andorra  Andorra la Vella      84000       468.0
1  United Arab Emirates         Abu Dhabi    4975593     82880.0
2           Afghanistan             Kabul   29121286    647500.0
3   Antigua and Barbuda        St. John's      86754       443.0
4              Anguilla        The Valley      13254       102.0
Columns: ['country', 'capital', 'Population', 'Area(Sq.Km)'], Rows: 250

🟡 LANGUAGES — HEAD:
           country,"No.","Official language"
0  Afghanistan,"2","Persian (Dari), Pashto"
1                    Albania,"1","Albanian"
2              Algeria,"2","Arabic, Berber"
3                     Andorra,"1","Catalan"
4                   Angola,"1","Portuguese"
Columns: ['country,"No.","Official language"'], Rows: 205

🟡 FOOD — HEAD:
                      country,"Food1","Food2","Food3"
0         Afghanistan,"Kabuli Pulao","Mantu","Ashak"
1              Albania,"Tavë Kosi","Byrek","Fërges

## Clean `country_languages.csv` file

In [179]:
# Reload the raw language CSV (in case previous load caused problems)
with open(files["languages"], "r", encoding="utf-8") as f:
    lang_lines = f.readlines()

# Strip whitespace & quotes, and split fields robustly
lang_rows = []
for line in lang_lines[1:]:  # Skip header
    parts = line.strip().split(',"')
    if len(parts) == 3:
        country = parts[0]
        no = parts[1].replace('"', '')
        official_language = parts[2].replace('"', '')
        lang_rows.append({
            "country": country,
            "official_language": official_language
        })
    else:
        print("⚠️ Parsing error in line:", line)

# Make DataFrame & normalise
lang_df = pd.DataFrame(lang_rows)
lang_df['country_clean'] = lang_df['country'].str.strip().str.lower()
lang_df = lang_df[['country_clean', 'official_language']]

# Remove unwanted quotes from country_clean (if any)
lang_df['country_clean'] = lang_df['country_clean'].str.replace('"', '').str.strip()


# Preview
print("\n[LANGUAGE CLEANED] — Sample:")
print(lang_df.head(8))
print("Columns:", lang_df.columns.tolist(), "Rows:", len(lang_df))



[LANGUAGE CLEANED] — Sample:
         country_clean                official_language
0          afghanistan           Persian (Dari), Pashto
1              albania                         Albanian
2              algeria                   Arabic, Berber
3              andorra                          Catalan
4               angola                       Portuguese
5  antigua and barbuda                          English
6            argentina  None (Spanishhasde factostatus)
7              armenia                         Armenian
Columns: ['country_clean', 'official_language'] Rows: 205


## Clean `country_food.csv` file

In [180]:
# Try to load the food CSV normally
try:
    food_df = pd.read_csv("raw/country_food.csv", sep=",", quotechar='"')
    print("[INFO] Columns loaded:", food_df.columns.tolist())
    if len(food_df.columns) == 1:
        raise ValueError("Only one column detected. Try fallback splitting.")
except Exception as e:
    print("[WARN] Problem with direct load:", e)
    # Fallback: Read without header, then split manually
    food_df = pd.read_csv("raw/country_food.csv", header=None)
    # Split the single column into 4 (country, Food1, Food2, Food3)
    split_cols = food_df[0].str.split(',', n=3, expand=True)
    food_df = pd.DataFrame(split_cols)
    food_df.columns = ['country', 'food1', 'food2', 'food3']
    # Drop the first row if it's a repeated header
    if food_df.iloc[0]['country'].lower().startswith('country'):
        food_df = food_df.iloc[1:]
    print("[INFO] Columns after manual fix:", food_df.columns.tolist())

# Remove any stray quotes and trim spaces
for col in ['country', 'food1', 'food2', 'food3']:
    food_df[col] = food_df[col].str.replace('"', '').str.strip()

# Preview
print("\n[FOOD CLEANED] — Sample:")
print(food_df.head(8))
print("Columns:", food_df.columns.tolist(), "Rows:", len(food_df))


[INFO] Columns loaded: ['country,"Food1","Food2","Food3"']
[WARN] Problem with direct load: Only one column detected. Try fallback splitting.
[INFO] Columns after manual fix: ['country', 'food1', 'food2', 'food3']

[FOOD CLEANED] — Sample:
               country                 food1         food2  \
1          Afghanistan          Kabuli Pulao         Mantu   
2              Albania             Tavë Kosi         Byrek   
3              Algeria              Couscous  Chakhchoukha   
4              Andorra             Escudella       Trinxat   
5               Angola     Muamba de Galinha        Calulu   
6  Antigua and Barbuda  Fungee and Pepperpot        Ducana   
7            Argentina                 Asado     Empanadas   
8              Armenia             Khorovats        Lavash   

                       food3  
1                      Ashak  
2                    Fërgesë  
3                    Makroud  
4       Cannelloni Andorrana  
5                      Funge  
6  Saltfish and

## Clean `country_flag.csv` file

In [181]:
# Load the flag CSV (normal read)
flag_df = pd.read_csv("raw/country_flag.csv")

# Clean up and rename columns
flag_df.columns = [col.strip().replace('"', '').lower().replace(' ', '_') for col in flag_df.columns]

# Only keep relevant columns: country and url
flag_df = flag_df[['country', 'url']].copy()

# Clean up quotes and whitespace
flag_df['country'] = flag_df['country'].str.replace('"', '').str.strip()
flag_df['url'] = flag_df['url'].str.replace('"', '').str.strip()

# Rename 'url' to 'flag_url'
flag_df = flag_df.rename(columns={'url': 'flag_url'})

# Lowercase and trim country for merging
flag_df['country_clean'] = flag_df['country'].str.strip().str.lower()

# Preview
print("\n[FLAGS CLEANED] — Sample:")
print(flag_df.head(8))
print("Columns:", flag_df.columns.tolist(), "Rows:", len(flag_df))



[FLAGS CLEANED] — Sample:
               country                                           flag_url  \
0          Afghanistan  https://www.worldometers.info//img/flags/small...   
1              Albania  https://www.worldometers.info//img/flags/small...   
2              Algeria  https://www.worldometers.info//img/flags/small...   
3              Andorra  https://www.worldometers.info//img/flags/small...   
4               Angola  https://www.worldometers.info//img/flags/small...   
5  Antigua and Barbuda  https://www.worldometers.info//img/flags/small...   
6            Argentina  https://www.worldometers.info//img/flags/small...   
7              Armenia  https://www.worldometers.info//img/flags/small...   

         country_clean  
0          afghanistan  
1              albania  
2              algeria  
3              andorra  
4               angola  
5  antigua and barbuda  
6            argentina  
7              armenia  
Columns: ['country', 'flag_url', 'country_clean'] Rows

## Merge all additional data 

In [182]:
# Ensure all DataFrames have 'country_clean' standardized
capitals['country_clean'] = capitals['country'].str.strip().str.lower()
lang_df['country_clean'] = lang_df['country_clean'].str.strip().str.lower()
food_df['country_clean'] = food_df['country'].str.strip().str.lower()
flag_df['country_clean'] = flag_df['country_clean'].str.strip().str.lower()

# Nimm nur die gewünschten Spalten aus capitals
merged = capitals[['country', 'capital', 'Population', 'country_clean']].copy()

# Merge Schritt für Schritt
merged = merged.merge(lang_df[['country_clean', 'official_language']], on='country_clean', how='left')
merged = merged.merge(food_df[['country_clean', 'food1', 'food2', 'food3']], on='country_clean', how='left')
merged = merged.merge(flag_df[['country_clean', 'flag_url']], on='country_clean', how='left')


# Preview und Save
print("\n[MERGED FINAL] — Sample:")
print(merged.head(10))
merged.to_csv("raw/countries_merged_final.csv", index=False)



[MERGED FINAL] — Sample:
                country           capital  Population         country_clean  \
0               Andorra  Andorra la Vella       84000               andorra   
1  United Arab Emirates         Abu Dhabi     4975593  united arab emirates   
2           Afghanistan             Kabul    29121286           afghanistan   
3   Antigua and Barbuda        St. John's       86754   antigua and barbuda   
4              Anguilla        The Valley       13254              anguilla   
5               Albania            Tirana     2986952               albania   
6               Armenia           Yerevan     2968000               armenia   
7                Angola            Luanda    13068161                angola   
8            Antarctica               NaN           0            antarctica   
9             Argentina      Buenos Aires    41343201             argentina   

                 official_language                 food1      food2  \
0                          Catala

## Final merge with all available data

In [183]:
# 1. Lade beide CSVs
cia = pd.read_csv("raw/countries_clean.csv")
merged = pd.read_csv("raw/countries_merged_final.csv")

# 2. (Optional) Whitespace entfernen, falls noch nicht gemacht
cia['country'] = cia['country'].str.strip()
merged['country'] = merged['country'].str.strip()

# 3. Merge: Alle Spalten von CIA + alle Spalten von Merged (Suffixe vermeiden Duplikate)
final = cia.merge(
    merged, 
    on="country", 
    how="left", 
    suffixes=("", "_x")   # Damit siehst du nachher sofort, falls doppelte Spalten entstehen
)

# 4. Doppelte Spalten bereinigen (z.B. wenn 'capital' oder so doppelt – behalte CIA-Variante)
final = final.loc[:,~final.columns.str.endswith('_x')]   # Entfernt alle Spalten mit Suffix "_x"

# 5. Zeilen entfernen, wo zu viele Spalten leer sind (z.B. wenn von Merged fast alles fehlt)
# Definiere die "wichtigen" Merged-Spalten
merged_important = [c for c in merged.columns if c != "country"]
min_fields_required = max(1, len(merged_important) // 2)  # Passe Threshold nach Bedarf an!

# Erstelle Maske, welche Zeilen genügend ausgefüllte Felder haben
enough_data = final[merged_important].notna().sum(axis=1) >= min_fields_required
final_clean = final[enough_data].copy()

# 6. Speichern & Preview
print(final_clean.head(10))
final_clean.to_csv("countries_final.csv", index=False)
print("✅ Merge complete. Final CSV ready!")



                country                                                url  \
0           Afghanistan  https://www.cia.gov/the-world-factbook/countri...   
2               Albania  https://www.cia.gov/the-world-factbook/countri...   
3               Algeria  https://www.cia.gov/the-world-factbook/countri...   
5               Andorra  https://www.cia.gov/the-world-factbook/countri...   
6                Angola  https://www.cia.gov/the-world-factbook/countri...   
9   Antigua and Barbuda  https://www.cia.gov/the-world-factbook/countri...   
10            Argentina  https://www.cia.gov/the-world-factbook/countri...   
11              Armenia  https://www.cia.gov/the-world-factbook/countri...   
14            Australia  https://www.cia.gov/the-world-factbook/countri...   
15              Austria  https://www.cia.gov/the-world-factbook/countri...   

                                           background  \
0   Ahmad Shah DURRANI unified the Pashtun tribes ...   
2   After declaring indepen